# ML-05 — Feature Vector and Leakage/Privacy Check

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

Plain Words Rule: If a content page has high staleness (content_age_days > 60) combined with a dropping trend in search performance, trigger a content refresh action to capture quick-win recovery.

Reason Code: STALE_CONTENT_REFRESH

Action Label: REFRESH_CONTENT

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [2]:
import os, getpass
import duckdb
import pandas as pd

HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your HF token: ')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'

query = f"""
SELECT
    content_hash_id,
    client_hash_id,
    'STALE_CONTENT_REFRESH' AS reason_code,
    'REFRESH_CONTENT' AS action_label,
    CAST(RANDOM() * 100 AS INTEGER) AS score
FROM '{REL}/fact_content_daily_performance/month=2026-03/*.parquet'
LIMIT 20
"""

df_queue = con.execute(query).df()

os.makedirs('work/outputs', exist_ok=True)
df_queue.to_csv('work/outputs/baseline_action_score.csv', index=False)

print("Successfully generated and saved baseline_action_score.csv!")
display(df_queue.head())

Paste your HF token: ··········
Successfully generated and saved baseline_action_score.csv!


,content_hash_id,client_hash_id,reason_code,action_label,score
0,content_b7e512995f79d5a6,client_73cda7b4e4f265ea,STALE_CONTENT_REFRESH,REFRESH_CONTENT,27
1,content_05597932fe4da067,client_73cda7b4e4f265ea,STALE_CONTENT_REFRESH,REFRESH_CONTENT,11
2,content_7a105f548d9c6916,client_73cda7b4e4f265ea,STALE_CONTENT_REFRESH,REFRESH_CONTENT,25
3,content_905aa32a0230694e,client_73cda7b4e4f265ea,STALE_CONTENT_REFRESH,REFRESH_CONTENT,85
4,content_a3ea9792f793ec72,client_73cda7b4e4f265ea,STALE_CONTENT_REFRESH,REFRESH_CONTENT,43


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

est Performed: Inspected feature correlations and verified that no label-derived columns or future performance windows (like next_month_traffic) are included in the predictor set.

Result: Clean separation achieved; features strictly rely on historical/current state variables available before the decision moment.

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

future_conversion_rate: Excluded to prevent direct target leakage.

post_query_click_share: Excluded because it represents post-decision outcomes rather than pre-decision inputs.

## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.